# Sondes linéaires, second tour

Même jeu, même séparation, même graine. Le premier tour (`sondes_lineaires.ipynb`) a retenu DINOv2-Base sur la validation (40 %), puis donné 37,5 % sur le test. Ce second tour, fixé avant de lancer, essaie trois pistes :
1. **DINOv2-Large** au lieu de Base ;
2. le vecteur global (CLS) **plus la moyenne des zones** de l'image, pour Base et Large ;
3. un entraînement **sans les variantes** (photos d'origine seulement) : le bruit de la recette d'origine replie les valeurs négatives et abîme ces images.

Le choix (réseau, vecteur, variantes ou non, C) se fait sur la validation seulement ; le test sert une fois, à la fin. Le test ayant déjà servi au premier tour, les deux résultats sont publiés.

In [1]:
import os, re, json
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

dataset_path = "dataset-v2"
seed = 12  # la graine du projet
img_size = (256, 256)
torch.manual_seed(seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

cuda NVIDIA GeForce GTX 1650


## 1. Rattacher chaque variante à sa photo d'origine

In [2]:
AUG = re.compile(r"^aug_(bright|flip|noise|rot|zoom)_")
class_names = sorted(d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)))

files = []  # (classe, fichier, photo d'origine)
for c in class_names:
    for f in sorted(os.listdir(os.path.join(dataset_path, c))):
        files.append((c, f, AUG.sub("", f)))

originals = {(c, f) for c, f, src in files if f == src}
assert all((c, src) in originals for c, f, src in files), "une variante sans photo d'origine"
print(len(files), "images,", len(originals), "photos d'origine,", len(class_names), "espèces")

4290 images, 715 photos d'origine, 13 espèces


## 2. Regrouper les photos d'origine presque identiques

Le MD5 du notebook de modélisation ne retire que les copies exactes. Une même photo enregistrée en `.jpg` et en `.png`, ou recadrée, passe au travers. On compare une empreinte visuelle (dHash 64 bits) de chaque paire de photos d'origine : à 6 bits d'écart ou moins, elles vont dans le même groupe, donc du même côté de la séparation.

In [3]:
def dhash(path):
    g = np.asarray(Image.open(path).convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    return (g[:, 1:] > g[:, :-1]).flatten()

orig = sorted(originals)
hashes = np.array([dhash(os.path.join(dataset_path, c, f)) for c, f in orig])
parent = list(range(len(orig)))
def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i

pairs = []
for i in range(len(orig)):
    d = (hashes[i + 1:] != hashes[i]).sum(axis=1)
    for j in np.flatnonzero(d <= 6) + i + 1:
        pairs.append((orig[i], orig[j], int(d[j - i - 1])))
        parent[root(j)] = root(i)

for a, b, d in pairs:
    print(f"{d:2d} bits  {a[0]}/{a[1]}  ~  {b[0]}/{b[1]}")
cross = [(a, b) for a, b, _ in pairs if a[0] != b[0]]
print(len(pairs), "paires presque identiques, dont", len(cross), "entre deux espèces différentes")
group = {o: root(i) for i, o in enumerate(orig)}

 6 bits  Chat/inat_735743851.jpg  ~  Lapin/inat_617482428.jpg
 5 bits  Lapin/inat_617482428.jpg  ~  Rat/inat_248440698.jpeg
2 paires presque identiques, dont 2 entre deux espèces différentes


## 3. Séparer par groupe, espèce par espèce

In [4]:
rng = np.random.default_rng(seed)
split = {}
for c in class_names:
    groups = sorted({g for (cc, _), g in group.items() if cc == c})
    rng.shuffle(groups)
    n_test = max(1, round(len(groups) * 0.15))
    n_val = max(1, round(len(groups) * 0.15))
    for k, g in enumerate(groups):
        split.setdefault(g, "test" if k < n_test else "val" if k < n_test + n_val else "train")

sets = {"train": [], "val": [], "test": []}
for c, f, src in files:
    s = split[group[(c, src)]]
    if s == "train" or f == src:  # les variantes augmentées ne vont qu'en entraînement
        sets[s].append((os.path.join(dataset_path, c, f), class_names.index(c)))

# Contrôle : aucun groupe, donc aucune photo d'origine, de part et d'autre de la séparation
grp = {os.path.join(dataset_path, c, f): group[(c, src)] for c, f, src in files}
g = {s: {grp[p] for p, _ in items} for s, items in sets.items()}
assert not (g["train"] & g["val"] or g["train"] & g["test"] or g["val"] & g["test"])
for s, items in sets.items():
    print(f"{s:5s} {len(items):5d} images", np.bincount([y for _, y in items], minlength=len(class_names)))

train  3036 images [234 234 234 234 234 234 234 234 234 234 228 234 234]
val     105 images [8 8 8 8 8 8 8 8 8 8 9 8 8]
test    104 images [8 8 8 8 8 8 8 8 8 8 8 8 8]


## Chargement

Même redimensionnement que les notebooks précédents (256 × 256, bilinéaire) ; chaque réseau applique ensuite son propre prétraitement.

In [5]:
imgs = {s: [Image.open(p).convert("RGB").resize(img_size, Image.BILINEAR) for p, _ in items] for s, items in sets.items()}
labels = {s: np.array([y for _, y in items]) for s, items in sets.items()}
print({s: len(v) for s, v in imgs.items()})

{'train': 3036, 'val': 105, 'test': 104}


## Caractéristiques de cinq réseaux pré-entraînés

Chaque image passe une fois dans le réseau gelé ; on garde son vecteur de sortie (avant toute tête de classement).

In [6]:
import torchvision.models as tvm
from transformers import AutoImageProcessor, AutoModel, CLIPVisionModel, SiglipVisionModel

def tv(builder, weights):
    m = builder(weights=weights).eval()
    if hasattr(m, "fc"): m.fc = torch.nn.Identity()   # ResNet
    if hasattr(m, "classifier"): m.classifier[-1] = torch.nn.Identity()   # ConvNeXt
    return m, weights.transforms(), lambda m, x: m(x)
def hf(cls, name):
    return cls.from_pretrained(name).eval(), AutoImageProcessor.from_pretrained(name), lambda m, x: m(pixel_values=x).pooler_output
def dino(name, mean):
    m = AutoModel.from_pretrained(name).eval()
    def fwd(m, x):
        o = m(pixel_values=x)   # pooler_output: the normalised CLS; last_hidden_state: CLS then one token per image patch
        return torch.cat([o.pooler_output, o.last_hidden_state[:, 1:].mean(1)], 1) if mean else o.pooler_output
    return m, AutoImageProcessor.from_pretrained(name), fwd
BACKBONES = {
    "DINOv2-Base CLS+zones": lambda: dino("facebook/dinov2-base", True),
    "DINOv2-Large": lambda: dino("facebook/dinov2-large", False),
    "DINOv2-Large CLS+zones": lambda: dino("facebook/dinov2-large", True),
}
@torch.no_grad()
def embed(name):
    m, prep, fwd = BACKBONES[name](); m.to(dev)
    out = {}
    for s, ims in imgs.items():
        chunks = []
        for i in range(0, len(ims), 32):
            x = prep(images=ims[i:i + 32], return_tensors="pt")["pixel_values"]
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
                chunks.append(fwd(m, x.to(dev)).float().cpu().numpy())
        out[s] = np.concatenate(chunks)
    del m; torch.cuda.empty_cache()
    return out

feats = {}
for name in BACKBONES:
    feats[name] = embed(name)
    np.savez(f"caracteristiques_{re.sub(r'[^A-Za-z0-9]+', '_', name)}.npz", **feats[name])
    print(f"{name:24s} {feats[name]['train'].shape[1]} caractéristiques")


Loading weights: 100%|##########| 223/223 [00:00<00:00, 2437.47it/s]
DINOv2-Base CLS+zones    1536 caractéristiques
Loading weights: 100%|##########| 439/439 [00:00<00:00, 2374.30it/s]
DINOv2-Large             1024 caractéristiques
Loading weights: 100%|##########| 439/439 [00:00<00:00, 2431.05it/s]
DINOv2-Large CLS+zones   2048 caractéristiques


## Choix du réseau et de la régularisation, sur la validation

In [7]:
y_tr, y_va, y_te = labels["train"], labels["val"], labels["test"]
orig_tr = np.array([not os.path.basename(f).startswith("aug_") for f, _ in sets["train"]])   # photos d'origine seulement
grid = {}
for name in BACKBONES:
    for variants in (True, False):
        keep = slice(None) if variants else orig_tr
        for C in (0.1, 0.3, 1, 3):
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
            clf.fit(feats[name]["train"][keep], y_tr[keep])
            grid[(name, variants, C)] = (accuracy_score(y_va, clf.predict(feats[name]["val"])), clf)
            print(f"{name:24s} variantes={'oui' if variants else 'non'} C={C:<4} validation {grid[(name, variants, C)][0]:.3f}")
best = max(grid, key=lambda k: grid[k][0])
print("choisi :", best, f"validation {grid[best][0]:.3f}")


DINOv2-Base CLS+zones    variantes=oui C=0.1  validation 0.333
DINOv2-Base CLS+zones    variantes=oui C=0.3  validation 0.314
DINOv2-Base CLS+zones    variantes=oui C=1    validation 0.352
DINOv2-Base CLS+zones    variantes=oui C=3    validation 0.371
DINOv2-Base CLS+zones    variantes=non C=0.1  validation 0.324
DINOv2-Base CLS+zones    variantes=non C=0.3  validation 0.324
DINOv2-Base CLS+zones    variantes=non C=1    validation 0.333
DINOv2-Base CLS+zones    variantes=non C=3    validation 0.314
DINOv2-Large             variantes=oui C=0.1  validation 0.400
DINOv2-Large             variantes=oui C=0.3  validation 0.419
DINOv2-Large             variantes=oui C=1    validation 0.419
DINOv2-Large             variantes=oui C=3    validation 0.400
DINOv2-Large             variantes=non C=0.1  validation 0.410
DINOv2-Large             variantes=non C=0.3  validation 0.419
DINOv2-Large             variantes=non C=1    validation 0.410
DINOv2-Large             variantes=non C=3    validatio

## Test, une seule fois, avec le couple choisi

In [8]:
name, variants, C = best
p = grid[best][1].predict(feats[name]["test"])
def bootstrap(y_true, y_pred, metric, n=2000):
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(y_true), (n, len(y_true)))
    return np.percentile([metric(y_true[i], y_pred[i]) for i in idx], [2.5, 97.5])
macro_f1 = lambda t, q: f1_score(t, q, average="macro", labels=range(len(class_names)), zero_division=0)
results = {"choix": {"reseau": name, "variantes": variants, "C": C, "validation": grid[best][0]},
           "validation_par_reseau": {f"{n} variantes={v} C={c}": r[0] for (n, v, c), r in grid.items()},
           "test": dict(n=len(y_te), accuracy=accuracy_score(y_te, p), f1_macro=macro_f1(y_te, p),
                        accuracy_ic95=bootstrap(y_te, p, accuracy_score).tolist(), f1_macro_ic95=bootstrap(y_te, p, macro_f1).tolist())}
src_test = np.array(["inat" if os.path.basename(f).startswith("inat_") else "origine" for f, _ in sets["test"]])
for s in ("origine", "inat"):
    m = src_test == s
    results["test_" + s] = dict(n=int(m.sum()), accuracy=accuracy_score(y_te[m], p[m]), accuracy_ic95=bootstrap(y_te[m], p[m], accuracy_score).tolist())
for k in ("choix", "test", "test_origine", "test_inat"): print(k, results[k])
print(classification_report(y_te, p, labels=range(len(class_names)), target_names=class_names, zero_division=0))
with open("resultats_sondes_tour2.json", "w") as f:
    json.dump(results, f, indent=2)

choix {'reseau': 'DINOv2-Large', 'variantes': True, 'C': 0.3, 'validation': 0.41904761904761906}
test {'n': 104, 'accuracy': 0.4807692307692308, 'f1_macro': 0.4823553771517573, 'accuracy_ic95': [0.38461538461538464, 0.5769230769230769], 'f1_macro_ic95': [0.3787532264115975, 0.5575722116913083]}
test_origine {'n': 48, 'accuracy': 0.4791666666666667, 'accuracy_ic95': [0.3333333333333333, 0.625]}
test_inat {'n': 56, 'accuracy': 0.48214285714285715, 'accuracy_ic95': [0.35714285714285715, 0.6071428571428571]}
              precision    recall  f1-score   support

      Castor       0.83      0.62      0.71         8
        Chat       0.30      0.38      0.33         8
       Chien       0.22      0.25      0.24         8
      Coyote       0.00      0.00      0.00         8
    Ecureuil       0.75      0.38      0.50         8
       Lapin       0.75      0.75      0.75         8
        Loup       0.30      0.38      0.33         8
        Lynx       0.40      0.25      0.31         8
   